# Customer Service Chatbot with LangGraph

LangGraph-powered chatbot managing support tickets via state machine. Routes conversations to specialized resolvers: positive/negative feedback handlers, status queries, and new ticket creation. Updates CSV database and generates contextual responses.

**Prerequisites:** Groq API key, test_data.csv, installed dependencies (requirements.txt)

In [ ]:
!pip install pandas
!pip install -U langchain-chroma
!pip install -U langchain_huggingface
!pip install -U sentence-transformers langchain.tools
!pip install langgraph
!pip install grandalf
!pip install langchain_graph groq langchain_groq
!pip install langgraph langchain_groq langchain_chroma langchain_huggingface pydantic typing-extensions

## State Management and LLM Configuration

Defines `State` TypedDict tracking messages, user_input, message_type, and ticket_number. Initializes ChatGroq with gpt-oss-120b model for classification, response generation, and summarization.

**Note:** Replace `<api_key>` with your Groq API key.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from typing import Annotated
from typing_extensions import TypedDict

class State(TypedDict):
    messages: Annotated[list[str], add_messages]
    user_input: str
    message_type: str
    ticket_number: int


from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, api_key="<api_key>")

## Message Classifier Schema

Pydantic model defining four message types: generic_query_resolver, positive_feedback_resolver, negative_feedback_resolver, new_ticket_generator. Ensures structured LLM output matching graph nodes.

In [ ]:
from typing import Literal


class MessageClassifier(BaseModel):
    message_type: Literal["generic_query_resolver", "positive_feedback_resolver", "negative_feedback_resolver", "new_ticket_generator"] = Field(description='message type')


## Positive Feedback Resolver

Handles resolved issues. Reads test_data.csv, updates ticket status to "Resolved", generates friendly acknowledgment. Returns response with updated ticket number.

In [ ]:
import os
import pandas as pd

def positive_feedback_resolver(state: State) -> str:
    '''
    Use this tool to provide a friendly response expressing gratitude towards the user upon successful resolution of their ticket
    '''
    print('Positive Feedback resolver invoked')

    ticket_number = state.get('ticket_number')

    system_prompt = f'\
                    Respond to the positive feedback received by the user. \
                    Greet them with a response indicating the pleasure of interacting with them and conducting business with them\
                    and indicate that ticket_number {ticket_number} has been marked as resolved'

    df = pd.read_csv('test_data.csv')

    df.loc[df['ticket_number']==state['ticket_number'][0], 'status']='Resolved'
    df.to_csv('test_data.csv', index=False)

    positive_feedback_prompt = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": state['messages'][0].content
        }
    ]

    result = llm.invoke(positive_feedback_prompt)
    return {"messages": [{"content": result.content, "role": "assistant"}], "ticket_number":ticket_number}


## New Ticket Generator

Creates tickets for new problems. Generates ticket number, uses LLM to summarize issue, creates database row with "Unresolved" status. Two LLM calls: summarization and response generation.

In [ ]:
import os
import pandas as pd

def new_ticket_generator(state: State) -> str:
    '''
    Use this tool to provide a friendly response expressing gratitude towards the user upon successful resolution of their ticket
    '''
    print('New ticket resolver invoked')

    df = pd.read_csv('test_data.csv')
    new_ticket_number = df['ticket_number'].astype(int).iloc[-1] + 1


    # We will need 2 llm calls here
    # 1. Summarize and update the existing status
    # 2. Provide a response to the user

    summarizer_system_prompt = """
                    Summarize {user_feedback} to create a new summary for the ticket
                    The summary should be less than 50 words
                    """

    df = pd.read_csv('test_data.csv')
    user_feedback=state['messages'][0].content

    summarizer_prompt = [
        {
            "role": "system",
            "content": summarizer_system_prompt
        },
        {
            "role": "user",
            "content": user_feedback
        }
    ]

    summary = llm.invoke(summarizer_prompt)

    system_prompt = f'\
                    Respond to the feedback received by the user. \
                    Greet them with a response indicating the pleasure of interacting with them and conducting business with them\
                    Analyze the feedback, if it is positive, provide a friendly greeting expressing gratitude for the feedback\
                    if feedback is negative\
                    and indicate that ticket_number:{new_ticket_number} has been created with this summary: {summary}\
                    if feedback cannot be classified, give a friendly message stating you are not able to process the query and ask how you can help'
    df.loc[len(df)] = [new_ticket_number, 'Unresolved', summary.content.replace("\n", "")]

    df.to_csv('test_data.csv', index=False)

    negative_feedback_prompt = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": state['messages'][0].content
        }
    ]

    result = llm.invoke(negative_feedback_prompt)
    return {"messages": [{"content": result.content, "role": "assistant"}], "ticket_number":new_ticket_number}


## Generic Query Resolver

Handles ticket status inquiries. Looks up ticket by number, retrieves status and notes, generates summary response. Returns friendly message if ticket not found.

In [ ]:
from collections import abc
import os
import pandas as pd

def generic_query_resolver(state: State) -> str:
    '''
    Use this tool to provide a friendly current status of ticket raised by user and express gratitude for their correspondence
    '''
    print('Generic Query resolver invoked')
    df = pd.read_csv('test_data.csv')
    ticket_number = state.get('ticket_number')
    current_row = df[df['ticket_number']==ticket_number]
    current_status = current_row['status']
    notes = current_row['notes']


    system_prompt = f'Greet with a friendly message acknowledging the user input For the ticket_number: {ticket_number}, current_status: {current_status} and notes: {notes} \
                    Generate a summary and provide a response\
                    if any of the values are missing simply provide a friendly message indicating no recordws for the ticket_number: {ticket_number} could be found'


    generic_query_prompt = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": state['messages'][0].content
        }
    ]

    result = llm.invoke(generic_query_prompt)
    return {"messages": [{"content": result.content, "role": "assistant"}], "ticket_number":ticket_number}


## Negative Feedback Resolver

Processes unresolved issues. Retrieves existing notes, combines with feedback, updates summary via LLM, sets status to "Unresolved". Two LLM calls: summarization and empathetic response.

In [ ]:
import os
import pandas as pd

def negative_feedback_resolver(state: State) -> str:
    '''
    Use this tool to provide a friendly response expressing gratitude towards the user upon successful resolution of their ticket
    '''
    print('Negative Feedback resolver invoked')

    # We will need 2 llm calls here
    # 1. Summarize and update the existing status
    # 2. Provide a response to the user

    summarizer_system_prompt = """
                    Summarize {user_feedback} and {current_notes} to create a new summary for the status of the ticket
                    The summary should be less than 50 words
                    """

    df = pd.read_csv('test_data.csv')
    user_feedback=state['messages'][0].content
    ticket_number = state.get('ticket_number')
    current_notes = df[df['ticket_number']==ticket_number]['notes'].item()
    summarizer_prompt = [
        {
            "role": "system",
            "content": summarizer_system_prompt
        },
        {
            "role": "user",
            "content": f'user_feedback: {user_feedback} current_notes: {current_notes}'
        }
    ]

    summarized_notes = llm.invoke(summarizer_prompt).content.replace("\n", "")
    df.loc[df['ticket_number']==ticket_number, 'status']='Unresolved'
    df.loc[df['ticket_number']==ticket_number, 'notes']=summarized_notes
    df.to_csv('test_data.csv', index=False)
    system_prompt = f'\
                    Analyze the negative feedback provider by customer in {user_feedback}, provide the summary you captured in the {summarized_notes}\
                    and provide a apologetic response indicating you are working earnestly to address their issue\
                    '
    negative_feedback_prompt = [
        {
            "role": "system",
            "content": system_prompt
        }
    ]

    result = llm.invoke(negative_feedback_prompt)
    return {"messages": [{"content": result.content, "role": "assistant"}], "ticket_number":state.get("ticket_number")}


## Router Function

Workflow entry point. Classifies messages into four types using MessageClassifier Pydantic model. Routes to appropriate resolver based on user intent and ticket presence.

In [ ]:
def router_message(state: State):
    user_input = state['messages'][0].content
    classifier_llm = llm.with_structured_output(MessageClassifier)
    ticket_number = state.get("ticket_number")
    system_prompt = """
                    Classify user message as 'positive_feedback_resolver', 'negative_feedback_resolver' or 'generic_query_resolver' based on the criteria below
                    'positive_feedback_resolver': if the user has provided input that indicates they are satisfied or happy with the resolution of their ticket, eg:  “Thanks for resolving my issue!"
                    'negative_feedback_resolver': if the user is still dissatisfied by the resolution Eg: “My problem still isn't fixed”
                    'generic_query_resolver': if user needs generic information regarding existing tickets or if they are seeking resolution for a brand new issue
                    'new_ticket_generator': if there is {ticket_number} is undefined/null and user is providing a brand new ticket
                    """

    final_prompt = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_input
        }
    ]


    result = classifier_llm.invoke(final_prompt)
    return {"messages": [{"content": result.message_type, "role":"assistant"}], "user_input":user_input, "message_type":result.message_type, "ticket_number":ticket_number}

## Final Response Formatter

Generates final user response after resolver processing. Combines query, ticket number, and context via LLM. Returns "No Context found" for misaligned context to prevent hallucination.

In [ ]:
def final_response(state: State):
    user_input = state["user_input"]
    context = state["messages"][-1].content # the last message is the final response
    ticket_number = state["ticket_number"]

    final_prompt = f'''
                    refer to the query and context below to build the response
                    if context is not aligned with the user query, the response must be 'No Context found'. Do not use
                    any additional context
                    user_query : {user_input}
                    ticket_number: {ticket_number}
                    context: {context}
                    '''
    result = llm.invoke(final_prompt)

    return {"messages": [{"content":result.content, "role":"assistant"}]}


## Graph Construction

Builds state machine orchestrating conversation flow. Nodes: resolvers, router, final_response. Edges: START→router, conditional routing to resolvers, all converge to final_response. ASCII visualization available.

In [ ]:
graph_builder = StateGraph(State)
graph_builder.add_node("router", router_message)
graph_builder.add_node("positive_feedback_resolver", positive_feedback_resolver)
graph_builder.add_node("negative_feedback_resolver", negative_feedback_resolver)
graph_builder.add_node("generic_query_resolver", generic_query_resolver)
graph_builder.add_node("new_ticket_generator", new_ticket_generator)

graph_builder.add_node("final_response", final_response)

graph_builder.add_edge(START, "router")
graph_builder.add_conditional_edges("router", lambda state: state.get("message_type"), {"positive_feedback_resolver": "positive_feedback_resolver", "negative_feedback_resolver": "negative_feedback_resolver", "generic_query_resolver": "generic_query_resolver", "new_ticket_generator": "new_ticket_generator"})
graph_builder.add_edge("positive_feedback_resolver", "final_response")
graph_builder.add_edge("negative_feedback_resolver", "final_response")
graph_builder.add_edge("new_ticket_generator", "final_response")
graph_builder.add_edge("generic_query_resolver", "final_response")

graph = graph_builder.compile()

graph.get_graph().print_ascii()


## Test Cases

Three test scenarios: negative feedback (ticket 1064), status query (ticket 1006), new ticket creation. Demonstrates routing, database updates, and response generation.

In [ ]:
result2 = graph.invoke({'messages':[{"role": "user", "content":"I am extremely displeased with your service"}], 'ticket_number':1064})
result2['messages']

In [ ]:
result3 = graph.invoke({'messages':[{"role": "user", "content":"what is the current status of this ticket"}], 'ticket_number':1006})
result3

In [ ]:
result4 = graph.invoke({'messages':[{"role": "user", "content":"My diecast car was never delivered"}]})
result4['messages'][-1]